# U-Net SAR Oil-Spill Trainer (GPU)

Trains the 2-channel VV/VH U-Net from `engines/detection/train.py` on a free GPU tier (Colab T4 / Kaggle P100). Run on GPU, then **download the `.pt` checkpoint** and run local CPU inference via `run_sar_end_to_end.py --model`.

**Workflow:** Drive mount -> data acquisition -> prepare -> train -> save/download.

> Everything in this repo already trains on CPU-only machines; this notebook simply runs the same `train.py` on a free GPU for practical speed.

## 0. Runtime
In Colab: **Runtime > Change runtime type > Hardware accelerator > GPU** (free T4). This cell verifies the GPU is visible.

In [ ]:
import torch, sys
print("Python", sys.version)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU detected - enable GPU runtime (Runtime>Change runtime type).")
DEVICE = 'cuda'

## 1. Mount Drive (Colab)
Saves the checkpoint to your Drive so it survives session restarts. On **Kaggle** skip this and use the `DATASET` input instead. You'll be prompted to authorise Drive access.

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/sih2026_models")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive mounted:", DRIVE_DIR)
except ImportError:
    # Not Colab (e.g. Kaggle) - write locally.
    DRIVE_DIR = Path("/content/out")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Not Colab; local out dir:", DRIVE_DIR)


## 2. Get the repo + install deps
Clone this project so `engines/detection/{train,prepare_dataset}.py` are available. If your repo is private/on GitHub, replace the URL. Kaggle: you can also just upload `engines/` into `/kaggle/working`.

In [ ]:
import os, subprocess, sys
ROOT = Path("/content/sih2026")
if not (ROOT / "engines"/ "detection" / "train.py").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/YOUR_ORG/sih2026-oil.git", str(ROOT)],
                    check=True)
os.chdir(ROOT)
!python -m pip install -q torch torchvision rasterio scipy matplotlib
print("cwd:", os.getcwd())


## 3. Data acquisition

Training source: the Zenodo **Sentinel-1 SAR Oil spill image dataset** (train/val/test images & binary masks):

* Part I  train+val images & masks: `https://zenodo.org/records/8346860`
* Part II val images & masks:       `https://zenodo.org/records/8253899`
* Part III test images & masks:     `https://zenodo.org/records/13761290`

Each archive has nested folders; `prepare_dataset.py` flattens them into `images/` + `masks/`. **Choose one source** by setting `DATA_MODE` below:

* `"zenodo"` - auto-download Zenodo archives (several GB; needs Drive storage).
* `"upload"` - you already uploaded a `S1_DATA` folder of TIFFs to Drive / a Kaggle dataset; it is flattened in place.

For a **quick smoke test** set `QUICK = True` -> uses a tiny synthetic pair. For real results set `QUICK = False`.

In [ ]:
DATA_MODE = "zenodo"       # "zenodo" | "upload"
QUICK = False              # True = tiny synthetic smoke test


In [ ]:
# --- SMOKE TEST (optional): build a 3-pair synthetic binary dataset ---
if QUICK:
    import numpy as np
    from PIL import Image
    d = Path("/content/s1data")
    (d/"images").mkdir(parents=True, exist_ok=True)
    (d/"masks").mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    for i in range(3):
        img = rng.normal(-30, 8, (64, 64)).astype("float32")
        msk = np.zeros((64, 64), "uint8")
        msk[16:48, 16:48] = 1
        Image.fromarray(img).save(d/"images"/f"{i:04d}.tif")
        Image.fromarray(msk).save(d/"masks"/f"{i:04d}.tif")
    SRC = d
    print("smoke dataset ready")


In [ ]:
if not QUICK and DATA_MODE == "zenodo":
    base = Path("/content/zenodo")
    base.mkdir(exist_ok=True)
    for name, rec in [
        ("train", "8346860"),
        ("val",   "8253899"),
        ("test",  "13761290"),
    ]:
        z = base / f"{name}.zip"
        if not z.exists():
            url = f"https://zenodo.org/records/{rec}/files/{z.name}"
            print(f"downloading {url}")
            !wget -q -O {z} "{url}"
        (base / name).mkdir(exist_ok=True)
        !unzip -q -o {z} -d {base/name}
    SRC = base
    print("zenodo archives extracted to", base)

elif not QUICK and DATA_MODE == "upload":
    # Point at a folder of TIFFs already in Drive or an uploaded dir.
    SRC = Path(input("Path to folder of image+mask TIFFs: ") or "/content/s1data")
    print("using upload dir", SRC)


## 4. Prepare `images/` + `masks/`
Flattens whatever was downloaded/uploaded into `data/datasets/s1_oil`.

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from engines.detection.prepare_dataset import main as prep_main
import argparse, sys as _s
out = Path("/content/data/datasets/s1_oil")
_s.argv = ["prepare_dataset.py", "--src", str(SRC), "--out", str(out), "--copy"]
prep_main()
print("prepared pairs in", out)
n_img = len(list((out/"images").glob("*.tif")))
n_msk = len(list((out/"masks").glob("*.tif")))
print(f"images={n_img} masks={n_msk}")
assert n_img > 0 and n_msk > 0, "no image/mask pairs prepared - inspect layout"

## 5. Train the U-Net on GPU
Runs `engines/detection/train.py`. For real data start with these params and raise `--epochs` once accuracy plateaus. `--patch 256 --base 32` fits easily on a T4. The best checkpoint (by val loss) is saved to `engines/detection/models/s1_unet.pt`.

In [ ]:
EPOCHS = 2 if QUICK else 60
BATCH = 16
PATCH = 64 if QUICK else 256
!python engines/detection/train.py \
    --data-dir /content/data/datasets/s1_oil \
    --epochs {EPOCHS} --batch-size {BATCH} --patch {PATCH} \
    --num-workers 2 --device cuda \
    --out /content/engines/detection/models/s1_unet.pt
print("training finished")


## 6. Save checkpoint to Drive + download
Copies the trained weights to Drive (permanent) and shows a download link (Colab) / copies into the Kaggle output directory. **This `.pt` file is what you copy back to your laptop** for local CPU inference.

In [ ]:
import shutil
ckpt = Path("/content/engines/detection/models/s1_unet.pt")
if ckpt.exists():
    dst = DRIVE_DIR / ckpt.name
    shutil.copyfile(ckpt, dst)
    print("saved to", dst, "| size MB:", round(ckpt.stat().st_size/1e6, 2))
    try:
        from google.colab import files
        files.download(str(ckpt))
    except ImportError:
        pass
else:
    print("checkpoint not found; check training output above")
print("\nNext step on your laptop:")
print("  python engines/detection/run_sar_end_to_end.py <incident> --model <s1_unet.pt>")


## 7. (Optional) Sanity-check inference on a test image
Loads the saved checkpoint and runs `UNet.predict` on the first test pair to confirm the weights load and produce a probability map.

In [ ]:
import numpy as np
import torch
from engines.detection.train import UNet
ckpt = torch.load("/content/engines/detection/models/s1_unet.pt",
                  map_location="cpu")
cfg = ckpt["config"]
model = UNet(in_channels=cfg["in_channels"], num_classes=cfg["num_classes"],
             base=cfg["base"])
model.load_state_dict(ckpt["model_state"])
model.eval()
print("checkpoint config:", cfg)
# quick fake image to prove forward pass works
fake = np.random.uniform(-40, 0, (PATCH, PATCH, 2)).astype("float32")
prob = __import__("engines.detection.train", fromlist=["predict"]).predict(
    model, fake, device=torch.device("cpu"), patch=PATCH)
print("prob map shape:", prob.shape, "range:", round(float(prob.min()),3), "-",
      round(float(prob.max()),3))
print("OK - model loads and predicts on CPU.")
